# Feature Engineering — Content Completion Prediction

Joins each viewing session with content + subscriber attributes and derives
pre-view context features. EXCLUDES post-view leakage (watch_duration_mins, rating).

**Reads:** `silver_viewing`, `silver_content`, `silver_subscribers`  **Writes:** `gold_ml_features`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp, when, dayofweek

spark = SparkSession.builder.getOrCreate()
print('Spark session ready')

In [ ]:
vh = spark.read.table('silver_viewing')
c = spark.read.table('silver_content')
s = spark.read.table('silver_subscribers')
print(f'views={vh.count():,} content={c.count():,} subscribers={s.count():,}')

required = {'view_id', 'subscriber_id', 'content_id', 'is_completed', 'view_hour'}
missing = required - set(vh.columns)
if missing:
    raise ValueError(f'silver_viewing missing columns {sorted(missing)}. Regenerate data and rerun bronze/silver.')

In [ ]:
# Join attributes + derive context features. EXCLUDE leakage (watch_duration_mins, rating).
# dayofweek(): 1=Sunday..7=Saturday -> map to 0=Mon..6=Sun and weekend flag.
ml_features = (
    vh.select(
        'view_id', 'subscriber_id', 'content_id', 'device_type', 'view_hour', 'view_date',
        col('is_completed').alias('had_complete'),
    )
    .join(c.select('content_id', 'genre', 'content_type', 'release_year',
                   'duration_mins', 'production_cost_bucket', 'language'),
          'content_id', 'left')
    .join(s.select('subscriber_id', 'plan_type', 'region', 'age_group', 'monthly_fee'),
          'subscriber_id', 'left')
    .withColumn('view_dow', ((dayofweek(col('view_date')) + 5) % 7))
    .withColumn('is_weekend', when(col('view_dow') >= 5, lit(1)).otherwise(lit(0)))
    .drop('view_date')
    .na.fill(0)
    .na.fill('unknown', subset=['genre', 'content_type', 'production_cost_bucket', 'language',
                                'plan_type', 'region', 'age_group', 'device_type'])
    .withColumn('feature_timestamp', current_timestamp())
)

total_rows = ml_features.count()
positive_rows = ml_features.filter(col('had_complete') == 1).count()
complete_rate = (positive_rows / total_rows * 100) if total_rows else 0.0

if total_rows < 1000 or positive_rows < max(10, int(total_rows * 0.01)):
    raise ValueError(
        f'Label quality check failed: only {positive_rows}/{total_rows} completed rows '
        f'({complete_rate:.2f}%). Check is_completed typing and source data.'
    )

ml_features.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_features')
print(f'Gold ML features written: {total_rows:,} rows | completion rate {complete_rate:.1f}%')

In [ ]:
spark.sql('OPTIMIZE gold_ml_features')
print('Feature table optimized')